<a href="https://colab.research.google.com/github/tavishgrade/nlp-language-model-portfolio/blob/main/Talktorial_2_Syntactic_TavishGrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Talktorial 2 — Syntactic: Dependency Parsing with spaCy

**Graduate NLP · Week 3 · Exposure #2** (lecture → **talktorial** → peer discussion)

In the lecture you saw that a word's context is its **structure**. Here you'll recover that
structure with a real parser, test the **PP-attachment** ambiguity hands-on, and check whether an
LLM parses the same way a dedicated parser does.

> ⚠️ **Colab users — do this first:** **File → Save a copy in Drive** (the runtime resets when idle).

**By the end you will be able to:**
1. Parse a sentence with **spaCy** and read its dependency tree
2. Visualize dependencies with **displaCy**
3. Show that **PP-attachment** needs *meaning*, not just syntax
4. Compare an **LLM's parse** to spaCy's — and verify it
5. See how a parser **breaks** on out-of-domain text

**Submit:** answer the prediction questions below, run all cells (outputs saved), complete the
**AI Verification Log**, answer the **reflection questions** at the end, then upload the
`.ipynb` to Canvas by **Sun 11:59 PM**.

## Before you code: make some predictions

Type your answers in this cell (double-click to edit):

- **Q1.** In *"I ate spaghetti with a **fork**"*, does *with* attach to the eating or the spaghetti?
  What about *"...with **meatballs**"*?
  > *Your answer:* The most probable assumption if I were deciding what to pair with what is that with a fork is referring to eating, while with meatballs is referring to the spaghetti. If you're asking what the parsing model will predict, I think both will be paired with spaghetti based on just the syntax.
- **Q2.** Will a parser trained on **news** handle a **tweet** well? Why or why not?
  > *Your answer:* Unless the news is about pop culture, I would assume it would have a smaller experiencial knowledge of coloquial verbiage. On top of that, the structure of published work is inherently a different structure than that of a tweet post so the modeling is framed on a different style and geared towards the content of the news.
- **Q3.** Will an **LLM** (ChatGPT/Claude) give the **same** dependency parse as spaCy?
  > *Your answer:* I assume from the lecture that LLMs and spaCy aren't entirely identical in backend structure. Even from last week we learned that simply identifying how you boundary your tokens can significantly change any modeling output. Assuming Claude isn't using spaCy, then absolutely the chance of identical dependency parse results is slim to none.

## Part 0 — Setup

Provided. Installs spaCy's small English model (safe to re-run after a reset).

In [ ]:
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

from spacy import displacy
print("spaCy ready:", nlp.meta["name"], nlp.meta["version"])

spaCy ready: core_web_sm 3.8.0


## Part 1 — Parse a sentence and read its structure

In [ ]:
# TODO: Parse "The dog chased the cat." with nlp(...).
# For each token, print: token text, tok.pos_, tok.dep_, and tok.head.text.
# (dep_ = the relation to its head; e.g. nsubj, dobj, det)

# parse through "The dog chased the cat" with spaCy parse
doc = nlp("The dog chased the cat.")

# iterate through each token and display variables
for tok in doc:
  print(tok.text, tok.pos_, tok.dep_, tok.head.text)


The DET det dog
dog NOUN nsubj chased
chased VERB ROOT chased
the DET det cat
cat NOUN dobj chased
. PUNCT punct chased


In [ ]:
# TODO: From `doc`, find the ROOT token (tok.dep_ == "ROOT"), the subject (nsubj),
# and the direct object (dobj). Print each. This is "who did what to whom".

# create empty variables to store sentence structure findings in
root = None
subject = None
dobj = None

# iterate through each sentence piece to find desired sentence structures
for tok in doc:
  if tok.dep_ == "ROOT":
    # if POS is the ROOT, save the token to the root variable
    root = tok
  elif tok.dep_ == "nsubj":
    # if POS is subject, save the token to the subject variable
    subject = tok
  elif tok.dep_ == "dobj":
    # if POS is direct object, save the token to the dobj variable
    dobj = tok

# my first hand at writing this script led to me forget the lagging _ in tok.dep_
# that made me realize that on another example my script here would break if a dobj was not present
# I've edited the following to provide a fail-safe for printing None variables, and to practice making this fail-safe

def safe_print(label, tok):
  if tok is None:
    print(f"the {label} is: none found")
  else:
    print(f"the {label} is: {tok.text}")

# safe print out name of each found token
safe_print("ROOT", root)
safe_print("subject", subject)
safe_print("direct object", dobj)

the ROOT is: chased
the subject is: dog
the direct object is: cat


## Part 2 — Visualize the dependency arcs (displaCy)

In [ ]:
# displaCy draws the labeled arcs. In Colab, this renders inline.
displacy.render(doc, style="dep", jupyter=True, options={"color": "brown","distance": 110})

## Part 3 — PP-attachment: when syntax needs meaning
The lecture's point: *same syntax, different attachment, decided by world knowledge.*

In [ ]:
# TODO: Parse "I ate spaghetti with a fork." and "I ate spaghetti with meatballs."
# For each, find the token "with" and print what it attaches to (with_tok.head.text).
# Did your Q1 prediction hold? Did spaCy get BOTH right?
for sent in ["I ate spaghetti with a fork.", "I ate spaghetti with meatballs."]:
  doc = nlp(sent)
  with_tok = [tok for tok in doc if tok.text == "with"][0]
  print(f"\nIn the sentence '{sent}' the word 'with' attaches to {with_tok.head.text}\n")
  # iterate through each token and display variables to double check results
  for tok in doc:
    print(tok.text, tok.pos_, tok.dep_, tok.head.text)

# My Q1 prediction was that both would attach to spaghetti based on what our lecture pushed
# I assumed the more common probability lies with the PP directly linking to what is before it
# I'm not sure that's the reason, and upon researching I see it could differ with spaCy versions



In the sentence 'I ate spaghetti with a fork.' the word 'with' attaches to spaghetti

I PRON nsubj ate
ate VERB ROOT ate
spaghetti NOUN dobj ate
with ADP prep spaghetti
a DET det fork
fork NOUN pobj with
. PUNCT punct ate

In the sentence 'I ate spaghetti with meatballs.' the word 'with' attaches to spaghetti

I PRON nsubj ate
ate VERB ROOT ate
spaghetti NOUN dobj ate
with ADP prep spaghetti
meatballs NOUN pobj with
. PUNCT punct ate


### ✅ Verification checkpoint #1
Ask your AI assistant: *"In dependency grammar, what does it mean for a prepositional phrase to
'attach' to a verb vs. a noun?"* Verify against the **Syntactic lecture**, **SLP3 §19.5 Ambiguity**
(where PP-attachment ambiguity is defined) and **SLP3 §20.1 Dependency Relations** (heads and
dependents). Careful: searching Ch. 20 for "attachment" finds *attachment accuracy*, which is an
evaluation metric and a different thing entirely. Record it in Part 6.

In [ ]:
# TODO: Parse "I shot the elephant in my pajamas." Find "in" and print its head.
# Which reading did spaCy choose -- the verb (shot) or the noun (elephant)?

ele_sent = nlp("I shot the elephant in my pajamas.")

# locate and store the details for the word 'in'
in_tok = [tok for tok in ele_sent if tok.text == "in"][0]

# print out all token information to verify my work
print(in_tok.text, in_tok.pos_, in_tok.dep_, in_tok.head.text)

print(f"\nIn the sentence '{ele_sent}' the word 'in' attaches to {in_tok.head.text}\n")

# spaCy chose the verb reading, which is interesting to me considering it chose the
# noun in the prior example. The result is in line with real meaning, unless this is a giant speaking, we don't know.

in ADP prep shot

In the sentence 'I shot the elephant in my pajamas.' the word 'in' attaches to shot



## Part 4 — Does an LLM parse like a parser?

1. In your browser, ask **ChatGPT or Claude**: *"Give the dependency parse (head + relation for each
   word) of: 'The old man the boats.'"* (a known garden-path sentence). Paste its answer below.
2. Run the cell to get **spaCy's** parse of the same sentence.
3. Compare them in the Verification Log (Part 6).

> *Paste the LLM's answer here:* ~~From ChatGPT on cognito -aka no prior bias with my previous discussions.~give the dependence parse (head + relation for each word) of: 'The old man the boats.'

ChatGPT said:
For “The old man the boats.”, this is the classic garden-path sentence where “man” is a verb meaning “to operate/staff.”

Word	Head	Relation

The	man	det

old	man	amod

man	man	ROOT

the	boats	det

boats	man	obj

So the dependency structure is:

        man (ROOT)
       /    \
    The     old
             \
             ?

More accurately, “The old” is the noun phrase functioning as the subject of “man”, while “the boats” is its object:

             man
           /     \
       old       boats
        |          |
       The        the

Thus:

man(The old, the boats)

The key ambiguity is that “old” is an adjective modifying the subject “The” + implicit noun? Actually, in the intended parse, “old” modifies “man” only if “man” is a noun, which is the misleading reading. In the correct parse, “The old” is the subject NP and “man” is the verb—making this sentence deliberately syntactically unusual.

In [ ]:
# spaCy's parse of the garden-path sentence, for comparison.
d = nlp("The old man the boats.")
for tok in d:
    print(f"{tok.text:6} {tok.pos_:6} {tok.dep_:10} -> {tok.head.text}")

The    DET    det        -> man
old    ADJ    amod       -> man
man    NOUN   ROOT       -> man
the    DET    det        -> boats
boats  NOUN   appos      -> man
.      PUNCT  punct      -> man


### ✅ Verification checkpoint #2
Did the LLM and spaCy agree? Did either handle the garden-path ("man" is the **verb**) correctly?
Verify your reasoning against the **Syntactic lecture**, the **spaCy docs**, and your own parse
above — and against **SLP3 §20.4 Evaluation**, which is where "was the parse correct?" gets a real
definition (labeled vs unlabeled attachment). Record it in Part 6.

## Part 5 — Where the parser breaks: domain shift

In [ ]:
# TODO: Parse an informal tweet (use the one given, or your own) and print POS/dep/head.
# Where does the news-trained parser get confused? Note 1-2 specific errors for Part 6.
tweet = "omg this restaurant was sooo good def coming back rn"

tweet_doc = nlp(tweet)

# iterate throught the tweet tokens and print out the details of POS, dep, and head
for tok in tweet_doc:
  print(tok.text, tok.pos_, tok.dep_, tok.head.text)

# iterating and saving details to a variable for the artifact
deps = [(tok.text, tok.pos_, tok.dep_, tok.head.text) for tok in tweet_doc]

# the news-trained parsed gets confused in a multitude of places
# omg is incorrectly listed as a noun when it's an interjection
# sooo is incorrectly listed as an adj dependent to def when it's an adverb modifying good

omg NOUN dep was
this DET det restaurant
restaurant NOUN nsubj was
was AUX ROOT was
sooo ADJ amod def
good ADJ amod def
def NOUN attr was
coming VERB acl def
back ADV advmod coming
rn ADP advmod was


## Part 6 — AI Verification Log

One block per checkpoint: the **question you asked**, the **claim** that came back, the **source** you
checked it against, your **verdict**, and a note saying what the check actually showed. Then add at
least one claim you chose yourself.

Recording the question matters as much as the claim. *"Isn't PMI just the log of the ratio?"* and
*"What does PMI compute?"* are different questions, and the first one mostly gets you agreement.

*(Double-click this cell to edit. Type after each bold label, then Shift+Enter to render it.)*

**A worked example, filled in so you can see the bar.** This one is not yours to answer:

- **What you asked:** "In spaCy, does the label `nsubj` always mark the subject of the sentence?"
- **What it told you:** Yes, `nsubj` marks the sentence's subject.
- **Source checked:** spaCy dependency-label docs, plus my own Part 1 parse
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** It marks the subject of *its own clause*, not of the sentence. A two-clause sentence has two
  `nsubj` tokens, each attached to its own verb — visible in my parse, which the one-sentence phrasing
  would have hidden.

---

### Claim 1 — checkpoint #1
- **What you asked:** In dependency grammar, what does it mean for a prepositional phrase to 'attach' to a verb vs. a noun?
- **What it told you:** attachment is which word is its head. It has to attach to something and it can be verb - action, or the noun - noun phrase.
- **Source checked:** SLP3Ch19.5 Ambiguity & SLP3Ch20.1, Ambiguity lecture on PP Attachment to VP/NP
- **Verdict:**⚠️
- **What you found when you checked:** While the words it said to me with its answer was correct, after verifying with the two chapters, a lot of details vital to the vernacular of this subject were missing. In regards to attachment of a PP, it said it could be either the action or the noun phrase, while the book *and most of the lecture/online reading* clearly use verb phrase and noun phrase, specifically as VP and NP. Leaving out that knowledge leaves a big gap for a new learner. As for head, the AI did describe attachment as 'which word is its head', but it left out that the attachment is then described as the dependent, another important vocabulary word of this field.
**However, the AI explanation went beyond the description in ch 19 when describing the two types of VP and NP attachment, classifying the attachment to the verb as 'instrumental attachment' and attachment to the noun phrase as 'nominal modification.'

### Claim 2 — checkpoint #2
- **What you asked:** ask AI to give the dependence parse (head + relation for each word) of: 'The old man the boats.' Did the LLM and spaCy agree? Did either handle the garden-path ("man" is the verb) correctly?
- **What it told you:** I used ChatGPT, and it gave both answers, incorrect and rare, to the garden-path example. It had the semantic knowledge of this exact example to first give me the classic garden-path dependency structure, but then goes on to correctly give the dependency structure with the word 'man' as the verb.
- **Source checked:** My spaCy output, the lecture, SLP3Ch20.4
- **Verdict:** ⚠️❌ claim does not hold for spaCy, and chatGPT and spaCy disagree
- **What you found when you checked:** Obviously the ChatGPT response does mention what the spaCy output came out with, but ultimately the ChatGPT response has the world knowledge it needs to correctly handle the garden-path example. The spaCy got caught up as the lecture told us it would by listing man as a noun instead of a verb. According to the SLP3 Ch 20.4 reading, spaCy's labeled attachment is incorrect for the head of "man," while ChatGPT's is correct. The two systems do not agree.

### Claim 3 — one you chose yourself
- **What you asked:** How does domain shift affect a news-trained parser's ability to correctly analyze informal text like tweets? Are the main error's more on POS tagging, dependency attachment, or both?
- **What it told you:** AI claimed both errors are prevelant, that slang or abbreviations would be easily misunderstood, and then further misattached in the dependency tree.
- **Source checked:** spaCy parse of a tweet, SLP3Ch.20.4, lecture on domain shift as an issue
- **Verdict:** ⚠️ It absolutely mentioned both as an issue, but it missed deeper details that should have been mentioned explained below.
- **What you found when you checked:** The spaCy parse verified what the AI output claimed, that both issues are prevelant and shown for the reasons they said. The abbreviation omg was mis labeled POS as a noun when its an interjection. Then the slang exaggerated sooo was mis labeled POS and mis attached as an adjective dependant to def when it's actually an adverb modifying good. The AI did catch those disctinctions, but missed the further detail of how we evaluate these parsings with metrics like LAS and UAS making a LS.

**Verdict key:** ✅ correct · ⚠️ partly right / needs caveat · ❌ wrong.

## Part 7 — Save your artifact (shared format → final project)

In [ ]:
# TODO: Build a dict `artifact` in the shared envelope:
#   {"week": 3, "task": "dependency-parsing", "source": ..., "outputs": {...}}
# Put the dependency triples (token, pos, dep, head) for one sentence in outputs.
# Save it as talktorial2_artifact.json.

# adding week and task as listed, then adding one sentence as source with its dependency triples as outputs
artifact = {
    "week": 3,
    "task": "dependency-parsing",
    "source": tweet,
    "outputs": {
        "dependencies": deps
    }
}

import json
with open("talktorial2_artifact.json", "w") as f:
  json.dump(artifact, f, indent=2)


## Before you submit — a few questions, honestly answered

These are not graded for being impressive. They are graded for being real, and they are
how I find out what is not landing while there is still time to do something about it.
*(Double-click this cell to type your answers, then Shift+Enter.)*

**1. What surprised you?**
*(Scroll back to your predictions at the top — where was your intuition wrong, and why?)*

> *Your answer:* I think because of what you covered in the lecture, nothing was surprising and I seemed to predict it correctly, but that doesn't mean I did it with any semblance of knowledge, just assumptions. I wouldn't say me being correct was because I knew why the spaCy parse had both fork and meatball attached to spaghetti, I couldn't tell you entirely correctly, but I understood the basic instinct of spaCy's difficulty with delineating these examples.

**2. What got you stuck this week, and what did you do about it?**
*("I am still stuck on X" is a complete and fully-credited answer.)*

> *Your answer:* Oof, well with the programming itself I felt less stuck, but with the details on why the model went one way for the elephant pj sentence and a different way for the spaghetti sentences, I couldn't give a clear explanation why that's happening. I think the issue I'm having the most is simply the English language. Math was always my declared favorite subject, so I'm actually getting tripped up on POS and sentence structure. The background idea that models do statistical things makes sense, it's just getting through the examples that's hard for me right now.

**3. What is one question you still have?**

> *Your answer:* In general, as I'm sifting through the SLP3 book for verificatino checks and otherwise, I see there's lots of nitty gritty details in there. Are you wanting us to read and know all the terms and details of these chapters listed that potentially go beyond what you cover in lecture?

---

### Working with AI

*There is no right answer here — "it frustrated me" and "I am not sure it helped" are as useful to me as anything positive. Answer honestly rather than helpfully.*

**a. Which AI did you use this week?** Name the assistant, and the model if you know it — e.g. "ChatGPT", "Claude", "Gemini", "Copilot", a local model, or "none this week", which is a perfectly good answer.

> *Your answer:* Copilot for technical clarification, ChatGPT for the specific moment you asked.

**b. After completing this week’s AI-supported task and checking the AI’s response, what did you think and feel about your experience with AI?**

*You might reflect on things such as what surprised you, what confused or frustrated you, what helped you learn, what made you feel more or less confident, or whether checking the AI’s response changed your thinking or feelings.*

> *Your answer:* I've always enjoyed the depth that AI's answers can go into sometimes. During the task it immediately called out the garden-path term and describe the typical pitfalls, when all I asked was verbatim what you said. I like when I get some background details to go along with a simple question. I usually try and reach for them when I phrase any sort of question, and even direct where I want extra information to come from so I can have some idea of its credibility.


## Wrap-up

You parsed real sentences, watched PP-attachment hinge on **meaning**, and saw both an LLM and a
news-trained parser stumble in different ways. Structure is everywhere — and, as the lecture said,
**context is everything.** Next up (**Week 5**, Semantics): structure → **meaning**.

**Optional extensions:** try Stanza (UD parses, many languages); parse a sentence in another language;
find a sentence where spaCy's PP-attachment is clearly wrong.

**Submit:** Run all cells, then **File → Download .ipynb** → upload to Canvas with the Verification
Log completed.